> 在线讲义：[第 7 章 · 把 Transformer 写成伪代码：和前作章节、a1 的函数一一对上](https://wikieden.github.io/beyond-transformer/lessons/07-formal-algorithms.html) · 本书《超越 Transformer》

# 第 7 章 · 把 Transformer 写成伪代码：和前作章节、a1 的函数一一对上

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/07-formal-algorithms.html) · 仓库里 `beyond-transformer/notes/lessons/07-formal-algorithms.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 对照表 | — | `MAP` | 10 行 |
| 第二步 Algorithm 3 | 缩放点积 + softmax | `single_query(e, ctx, ...)` | α = [0.4458, 0.1084, 0.4458]，输出 [0.8916, −0.5542] |
| 第三步 Algorithm 10 的形状 | — | `shape_trace()` | 论文 (4, 5)、a1 (5, 4)；输出 (10, 5) / (5, 10) |

Algorithm 3（单 query 注意力）算一个小例子；Algorithm 10（DTransformer）每一行的形状（论文是“列 = token”，a1 是“行 = token”）。
只跟踪形状、算一个单 query 的例子；完整的 a1 §3 前向留给 T-M0。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MAP = [  # (算法, 做什么, 前作章, a1 里的题)
    ("Algorithm 1", "Token embedding：查表取一列", "T22", "embedding"),
    ("Algorithm 2", "Positional embedding：位置也查表（或用 sin/cos）", "T14–T16", "rope（a1 换成 RoPE）"),
    ("Algorithm 3", "单 query 注意力：一个 token 看全部上下文", "T2–T9", "softmax、scaled_dot_product_attention"),
    ("Algorithm 4", "整段注意力 + Mask", "T11、T12、T17", "scaled_dot_product_attention"),
    ("Algorithm 5", "多头注意力", "T13", "multihead_self_attention"),
    ("Algorithm 6", "LayerNorm", "T19", "rmsnorm（a1 换成 RMSNorm）"),
    ("Algorithm 7", "Unembedding：乘回词表再 softmax", "T22", "transformer_lm 的输出层"),
    ("Algorithm 10", "DTransformer：GPT 式的整体前向", "T21、T24", "transformer_block、transformer_lm"),
    ("Algorithm 13", "DTraining：交叉熵 + 梯度下降", "第 1–4 章", "cross_entropy、adamw、training_together"),
    ("Algorithm 14", "DInference：逐个采样生成", "T22", "decoding"),
]

**函数 `single_query`**：Algorithm 3：q = Wq e；k_t = Wk e_t；v_t = Wv e_t；α = softmax(qᵀk / √d_attn)；输出 Σ α_t v_t。

In [ ]:
def single_query(e, ctx, wq, wk, wv):
    """Algorithm 3：q = Wq e；k_t = Wk e_t；v_t = Wv e_t；α = softmax(qᵀk / √d_attn)；输出 Σ α_t v_t。"""
    q = wq @ e
    k = ctx @ wk.T
    v = ctx @ wv.T
    s = k @ q / np.sqrt(len(q))
    a = np.exp(s - s.max())
    a = a / a.sum()
    return s, a, a @ v

In [ ]:
E_CUR = np.array([1.0, 0.0])
CTX = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
WQ = np.array([[2.0, 0.0], [0.0, 2.0]])
WK = np.eye(2)
WV = np.array([[1.0, 0.0], [0.0, -1.0]])

**函数 `shape_trace`**：Algorithm 10 每一行之后的形状：论文写法（d × n，一列一个 token） vs a1 写法（n × d，一行一个 token）。

In [ ]:
def shape_trace(V=10, L=2, H=2, de=4, dmlp=16, n=5):
    """Algorithm 10 每一行之后的形状：论文写法（d × n，一列一个 token） vs a1 写法（n × d，一行一个 token）。"""
    rows = [("行 2 嵌入：W_e[:, x[t]] + W_p[:, t]", (de, n)), ("行 3 拼成 X", (de, n))]
    for l in range(1, L + 1):
        rows += [(f"第 {l} 层 行 5 layer_norm", (de, n)), (f"第 {l} 层 行 6 X + MHAttention（{H} 头，每头 {de // H} 维）", (de, n)),
                 (f"第 {l} 层 行 7 layer_norm", (de, n)), (f"第 {l} 层 行 8 MLP 中间（GELU）", (dmlp, n)), (f"第 {l} 层 行 8 X + MLP", (de, n))]
    rows += [("行 10 最后的 layer_norm", (de, n)), ("行 11 W_u X → logits", (V, n)), ("行 11 softmax → P", (V, n))]
    return [(name, shp, shp[::-1]) for name, shp in rows]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 算法 ↔ 前作章节 ↔ a1")
for a, what, t, a1 in MAP:
    print(f"  {a:12s} {what}  | {t} | {a1}")
print("\n第二步 Algorithm 3：单 query 注意力")
s, a, out = single_query(E_CUR, CTX, WQ, WK, WV)
print(f"  q = {(WQ @ E_CUR).tolist()}；分数 qᵀk/√2 = {np.round(s, 4).tolist()}；α = {np.round(a, 4).tolist()}；输出 = {np.round(out, 4).tolist()}")

In [ ]:
print("\n第三步 Algorithm 10 的形状（V = 10、L = 2、H = 2、d_e = 4、d_mlp = 16、序列 5）")
for name, paper, a1 in shape_trace():
    print(f"  {name:42s} 论文 {paper}  a1 {a1}")

## 3. 代码里才看得到的

- **同一个函数里两种写法都出现了**：q 用列向量左乘，k、v 因为上下文按行存，用右乘 Wᵀ。写 a1 时统一成“一行一个 token、右乘 Wᵀ”最不容易错。
- **这里只跟踪形状、算一个单 query 的例子**：a1 §3 的完整前向（带参数、能过测试）留给里程碑 T-M0。

## 4. 自己改一改

- 把 `single_query` 的 $W_q$ 改成 5I，α 怎么变？（分数放大，softmax 更尖）
- 给 `shape_trace` 传 L = 4，表多出几行？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 十四个算法一本说明书：一到七是零件，十是 GPT，十三训练、十四生成。
> 单个 query 看上下文：缩放点积、softmax、加权 value。
> 并排做加 mask 是四，做多头是五。
> 先 norm 再子层，结果加回 X；只有 MLP 中间变宽。
> 论文一列一 token，a1 一行一 token，形状互为转置。